# TF-IDF (Term Frequency - Inverse Document Frequency)
This notebook visualizes how search queries are ranked against business profiles in the Acuity system.
TF-IDF calculates the importance of a word in a document relative to its frequency across all documents.



In [ ]:
import math
import re
from collections import defaultdict
import pandas as pd
from IPython.display import display

# Standard English stop words
STOP_WORDS = set([
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", 
    "yours", "yourself", "yourselves", "he", "him", "his", "himself", "she", 
    "her", "hers", "herself", "it", "its", "itself", "they", "them", "their", 
    "theirs", "themselves", "what", "which", "who", "whom", "this", "that", 
    "these", "those", "am", "is", "are", "was", "were", "be", "been", "being", 
    "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an", 
    "the", "and", "but", "if", "or", "because", "as", "until", "while", "of", 
    "at", "by", "for", "with", "about", "against", "between", "into", "through", 
    "during", "before", "after", "above", "below", "to", "from", "up", "down", 
    "in", "out", "on", "off", "over", "under", "again", "further", "then", 
    "once", "here", "there", "when", "where", "why", "how", "all", "any", 
    "both", "each", "few", "more", "most", "other", "some", "such", "no", 
    "nor", "not", "only", "own", "same", "so", "than", "too", "very", "s", 
    "t", "can", "will", "just", "don", "should", "now"
])

class CustomTfidfVectorizer:
    def __init__(self, ngram_range=(1, 2)):
        self.ngram_range = ngram_range
        self.idf_weights = {}
        self.vocabulary = set()
        self.n_documents = 0
    
    def _tokenize_and_ngrams(self, text):
        text = text.lower()
        tokens = re.findall(r'\b[a-z0-9]+\b', text)
        tokens = [t for t in tokens if t not in STOP_WORDS]
        ngrams = []
        min_n, max_n = self.ngram_range
        if min_n <= 1:
            ngrams.extend(tokens)
        for n in range(max(2, min_n), max_n + 1):
            for i in range(len(tokens) - n + 1):
                ngrams.append(" ".join(tokens[i:i+n]))
        return ngrams

    def fit_transform(self, documents):
        self.n_documents = len(documents)
        document_term_counts = []
        doc_frequency = defaultdict(int)
        
        for doc in documents:
            tokens = self._tokenize_and_ngrams(doc)
            term_counts = defaultdict(int)
            for token in tokens:
                term_counts[token] += 1
            document_term_counts.append(term_counts)
            for unique_term in set(tokens):
                doc_frequency[unique_term] += 1
                self.vocabulary.add(unique_term)
                
        for term, df in doc_frequency.items():
            self.idf_weights[term] = math.log(self.n_documents / df)
            
        tfidf_vectors = []
        for term_counts in document_term_counts:
            vector = {}
            for term, count in term_counts.items():
                if count > 0:
                    tf = 1.0 + math.log(count)
                    vector[term] = tf * self.idf_weights[term]
            tfidf_vectors.append(vector)
        return tfidf_vectors

    def transform(self, documents, partial_match=False):
        tfidf_vectors = []
        for doc in documents:
            tokens = self._tokenize_and_ngrams(doc)
            term_counts = defaultdict(int)
            for token in tokens:
                if token in self.vocabulary:
                    term_counts[token] += 1
                if partial_match and len(token) >= 3:
                    for vocab_term in self.vocabulary:
                        if vocab_term != token and (vocab_term.startswith(token) or token.startswith(vocab_term)):
                            term_counts[vocab_term] += 1
            
            vector = {}
            for term, count in term_counts.items():
                if count > 0:
                    tf = 1.0 + math.log(count)
                    vector[term] = tf * self.idf_weights[term]
            tfidf_vectors.append(vector)
        return tfidf_vectors

# Mock Business Data
documents = [
    "Coffee shop serving hot espresso, lattes, and fresh pastries.",
    "Hardware store selling tools, lumber, and home repair supplies.",
    "Computer repair shop specializing in laptops and hardware fixing.",
    "Boutique selling vintage clothing, dresses, and accessories.",
    "Cozy cafe and bakery with espresso and vintage decor."
]
business_names = ["Joe's Coffee", "Bob's Hardware", "Tech Fixers", "Retro Threads", "The Vintage Cafe"]

# 1. Initialize and fit the Custom TF-IDF Vectorizer
vectorizer = CustomTfidfVectorizer(ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(documents)

# 2. Visualize the Dictionary Weights
df_tfidf = pd.DataFrame(tfidf_matrix, index=business_names).fillna(0)

print("TF-IDF Weights (Sample of terms across businesses):")
display(df_tfidf[['espresso', 'hardware', 'vintage', 'coffee', 'repair']])

### Query Cosine Similarity
Now we simulate a user typing a search query, vectorizing it into the same TF-IDF space, and measuring the cosine similarity (angle between the vectors) to find the most relevant businesses.



In [ ]:
# User Search Query
user_query = "vintage espresso cafe"

# Vectorize the query using partial_match=True
query_vecs = vectorizer.transform([user_query], partial_match=True)
query_vec = query_vecs[0]

# Calculate Cosine Similarity between query and all documents
def cosine_sim(vec1, vec2):
    dot = sum(vec1.get(k, 0) * vec2.get(k, 0) for k in set(vec1) | set(vec2))
    mag1 = math.sqrt(sum(v**2 for v in vec1.values()))
    mag2 = math.sqrt(sum(v**2 for v in vec2.values()))
    return dot / (mag1 * mag2) if mag1 and mag2 else 0.0

similarities = [cosine_sim(query_vec, doc_vec) for doc_vec in tfidf_matrix]

# Display Ranked Results
results = pd.DataFrame({
    'Business': business_names,
    'Relevance Score': similarities
}).sort_values(by='Relevance Score', ascending=False)

print(f"Search Query: '{user_query}'\n")
print("Ranked Results:")
display(results)